# Stream a Response via our Bedrock Platform

This example shows how to stream a response from the Bedrock platform using our platform client.

## Setup


In [1]:
import os

import dotenv

from agent_server_types_v2.platforms.bedrock import BedrockPlatformParameters
from agent_server_types_v2.platforms.bedrock.configs import BedrockModelMap

# Env
dotenv.load_dotenv()

masked_key_id = os.getenv("AWS_ACCESS_KEY_ID")[:5] + "*" * (
    len(os.getenv("AWS_ACCESS_KEY_ID")) - 5
)
print(f"Using AWS Credentials: {masked_key_id}")


# Configurations that will be used
default_model_map = BedrockModelMap.default()
BedrockModelMap.set_instance(default_model_map)

# Platform Parameters
parameters = BedrockPlatformParameters(
    region_name=os.getenv("AWS_DEFAULT_REGION"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)


Using AWS Credentials: AKIA4***************


## Create a Platform Client

In [2]:
from agent_server_types_v2.platforms.bedrock.client import BedrockClient

bedrock_client = BedrockClient(
    parameters=parameters,
)

## Create a Tool

In [3]:
from typing import Annotated

from agent_server_types_v2.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False

joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)



ToolDefinition(name='joke_judge', description='Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', input_schema={'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}, function=<function joke_judge at 0x7fecbc5de0c0>)


## Create a Prompt

In [4]:
from agent_server_types_v2.prompts import PromptTextContent, PromptUserMessage
from agent_server_types_v2.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[PromptTextContent(
        text="Please use the joke_judge tool to "
        "judge if the following joke is funny: "
        "\"Why did the chicken cross the road?\"\n"
        "\"To get to the other side!\"",
        )],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool],
)
bedrock_prompt = bedrock_client.converters.convert_prompt(general_prompt)

print(bedrock_prompt)



BedrockPrompt(messages=[{'role': 'user', 'content': [{'text': 'Please use the joke_judge tool to judge if the following joke is funny: "Why did the chicken cross the road?"\n"To get to the other side!"'}]}], system=[{'text': 'You are a helpful assistant.'}], inference_config=None, tool_config={'tools': [{'toolSpec': {'name': 'joke_judge', 'description': 'Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', 'inputSchema': {'json': {'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}}}}], 'toolChoice': {'auto': {}}}, guardrail_config=None, additional_model_request_fields=None, prompt_variables=None, additional_model_response_field_paths=None, request_metadata=None, performance_config=None)


## Choose the Model

In this case, we will create a ModelSelector that returns the model we want, but normally, we would call the selector attached to the kernel and accessible via `kernel.platform.get_model`. In the full implementation, the server is responsible to check if `kernel.agent.agent_architecture.get_model` returns a model first before calling the platform's default selector.

In [5]:
from agent_server_types_v2.model_selector import ModelSelector
from agent_server_types_v2.models.model import Model, Models


class Claude35SonnetModelSelector(ModelSelector):
    def select_model(self, selection: str | None = None) -> Model:
        return Models.ANTHROPIC_CLAUDE_3_5_SONNET

model_selector = Claude35SonnetModelSelector()



## Request and Stream a Response

In [6]:
deltas = []
i = 0
async for delta in bedrock_client.generate_stream_response(bedrock_prompt, model_selector):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1


CHUNK 0: GenericDelta(op='add', path='/role', value='agent', from_=None)
CHUNK 1: GenericDelta(op='add', path='/content', value=[], from_=None)
CHUNK 2: GenericDelta(op='add', path='/content/0', value={'kind': 'text', 'text': 'I'}, from_=None)
CHUNK 3: GenericDelta(op='concat_string', path='/content/0/text', value="'ll help you judge if this", from_=None)
CHUNK 4: GenericDelta(op='concat_string', path='/content/0/text', value=' classic joke is funny using', from_=None)
CHUNK 5: GenericDelta(op='concat_string', path='/content/0/text', value=' the joke_judge', from_=None)
CHUNK 6: GenericDelta(op='concat_string', path='/content/0/text', value=" tool. I'll combine", from_=None)
CHUNK 7: GenericDelta(op='concat_string', path='/content/0/text', value=' both parts', from_=None)
CHUNK 8: GenericDelta(op='concat_string', path='/content/0/text', value=' of the joke into', from_=None)
CHUNK 9: GenericDelta(op='concat_string', path='/content/0/text', value=' one string.', from_=None)
CHUNK 10: Ge

## Reassemble

We now try to reassemble to full message to ensure it properly assembles into a `ResponseMessage`.

An implementation of this will need to be either in the AA or in a helper method attached to the kernel's `PlatformInterface`.

### First as a Dictionary

In [7]:
from pprint import pprint

from agent_server_types_v2.delta.combine_delta import combine_generic_deltas

assembeld_dict = combine_generic_deltas(deltas)
pprint(assembeld_dict)

{'content': [{'kind': 'text',
              'text': "I'll help you judge if this classic joke is funny using "
                      "the joke_judge tool. I'll combine both parts of the "
                      'joke into one string.'},
             {'kind': 'tool_use',
              'tool_call_id': 'tooluse_7wtea46yTByvKCYWIX8hhA',
              'tool_input_raw': '{"joke": "Why did the chicken cross the road? '
                                'To get to the other side!"}',
              'tool_name': 'joke_judge'}],
 'metadata': {'host_id': None,
              'http_headers': {'connection': 'keep-alive',
                               'content-type': 'application/vnd.amazon.eventstream',
                               'date': 'Thu, 27 Feb 2025 18:52:25 GMT',
                               'transfer-encoding': 'chunked',
                               'x-amzn-requestid': '3de784cf-6768-4b8e-9214-455654b33d2c'},
              'http_status_code': 200,
              'request_id': '3de784cf-

### Now as a ResponseMessage

In [8]:
from agent_server_types_v2.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembeld_dict)
pprint(response_message)




ResponseMessage(content=[ResponseTextContent(kind='text',
                                             uncommitted_deltas=[],
                                             text="I'll help you judge if this "
                                                  'classic joke is funny using '
                                                  "the joke_judge tool. I'll "
                                                  'combine both parts of the '
                                                  'joke into one string.'),
                         ResponseToolUseContent(kind='tool_use',
                                                uncommitted_deltas=[],
                                                tool_call_id='tooluse_7wtea46yTByvKCYWIX8hhA',
                                                tool_name='joke_judge',
                                                tool_input_raw='{"joke": "Why '
                                                               'did the '
                     

# Continue the Conversation

## Execute the Tool

In [9]:
from agent_server_types_v2.responses import ResponseToolUseContent

tool_use_content_block = response_message.content[1]
assert isinstance(tool_use_content_block, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

Is the joke funny? No


## Create a ToolResponse Content Block

> **Note:** This skips writing to a thread, but we convert the tool content block of the `ResponseMessage` to a `ThreadToolUsageContent` so we can convert to `PromptToolResultContent`.

In [10]:
from agent_server_types_v2.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block, ResponseToolUseContent)
tool_result_content = PromptToolResultContent(
    tool_name=tool_use_content_block.tool_name,
    tool_call_id=tool_use_content_block.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

print(tool_result_content)


PromptToolResultContent(kind='tool_result', tool_name='joke_judge', tool_call_id='tooluse_7wtea46yTByvKCYWIX8hhA', content=[PromptTextContent(kind='text', text='False')], is_error=False)


## Extract the AI's Tool Usage Content Block

We must send back the AI's original tool use in the new request

In [11]:
from agent_server_types_v2.prompts import PromptToolUseContent
from agent_server_types_v2.thread import ThreadToolUsageContent

original_tool_use = response_message.content[1]
assert isinstance(original_tool_use, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use = ThreadToolUsageContent.from_response_tool_use(original_tool_use)
prompt_tool_use = PromptToolUseContent.from_thread_tool_usage(ai_tool_use)

pprint(prompt_tool_use)


PromptToolUseContent(kind='tool_use',
                     tool_call_id='tooluse_7wtea46yTByvKCYWIX8hhA',
                     tool_name='joke_judge',
                     tool_input_raw='{"joke": "Why did the chicken cross the '
                                    'road? To get to the other side!"}',
                     _tool_input={'joke': 'Why did the chicken cross the road? '
                                          'To get to the other side!'})


## Create new Prompt

In [12]:
from agent_server_types_v2.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool],
)
return_bedrock_prompt = bedrock_client.converters.convert_prompt(return_gen_prompt)

pprint(return_bedrock_prompt)

BedrockPrompt(messages=[{'content': [{'text': 'Please use the joke_judge tool '
                                              'to judge if the following joke '
                                              'is funny: "Why did the chicken '
                                              'cross the road?"\n'
                                              '"To get to the other side!"'}],
                         'role': 'user'},
                        {'content': [{'toolUse': {'input': {'joke': 'Why did '
                                                                    'the '
                                                                    'chicken '
                                                                    'cross the '
                                                                    'road? To '
                                                                    'get to '
                                                                    'the other '
                   

## Generate a new Response (Non-Streaming)

In [14]:
response = await bedrock_client.generate_response(return_bedrock_prompt, model_selector)

pprint(response)

ResponseMessage(content=[ResponseTextContent(kind='text',
                                             uncommitted_deltas=[],
                                             text='According to the joke_judge '
                                                  'tool, this joke is not '
                                                  'considered funny. This is '
                                                  'perhaps not surprising, as '
                                                  'the "Why did the chicken '
                                                  'cross the road?" joke is '
                                                  'often considered one of the '
                                                  'most basic and overused '
                                                  'jokes, with a very simple '
                                                  'and predictable punchline. '
                                                  'The humor is supposed to '
       